# ai software delivery (use case 2)
An automated software engineering pipeline featuring agents for requirements analysis, system architecture, modular coding, testing, and unified documentation.

### step 1: install dependencies
Install the latest version of **CrewAI** and its components for requirement analysis and system design.

In [1]:
# Install dependencies if not already installed
!pip install crewai langchain_community python-docx


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### step 2: initialization & imports
Load environment variables and necessary modules to start the agentic workflow.

In [2]:
import os
from crewai import Agent, Task, Crew, Process

# Set OpenAI API Key - Replace with your key or load from environment
# os.environ["OPENAI_API_KEY"] = "your_api_key"
# If using .env file:
from dotenv import load_dotenv
load_dotenv() # Adjust path as needed or just load .env

True

### step 3: define requirement analyst agent & task
This agent focuses on translating high-level feature requests into detailed documentation.

In [3]:
# 1. Define Requirement Analyst Agent
requirement_analyst = Agent(
    role='Product Analyst',
    goal='Convert natural language feature into structured technical specification',
    backstory='Experienced product manager translating business needs to tech specs',
    verbose=True,
    allow_delegation=False
)

In [4]:
# 2. Define Requirement Analysis Task
requirement_analysis_task = Task(
    description='Analyze the following feature request and generate a detailed technical specification: {feature_request}',
    expected_output='A structured technical specification including:\n'\
                    '1. Functional Requirements\n'\
                    '2. Non-functional Requirements\n'\
                    '3. Acceptance Criteria',
    agent=requirement_analyst,
    human_input=True # Enable human feedback loop
)

### step 4: define software architect agent & task
Responsible for designing the overarching system architecture and technical stack recommendations.

In [5]:
# 3. Define Software Architect Agent
software_architect = Agent(
    role='System Architect',
    goal='Design system architecture and folder structure based on technical specifications',
    backstory='10+ years designing scalable backend systems and organizing large codebases',
    verbose=True,
    allow_delegation=False
)

In [6]:
# 4. Define Architecture Task
architecture_task = Task(
    description='Based on the provided technical specification, design the system architecture.',
    expected_output='A detailed architecture document including:\n'\
                    '1. Architecture Overview (Diagram description or text)\n'\
                    '2. Tech Stack Recommendation with justification\n'\
                    '3. File and Module Breakdown (Folder structure) with root folder\n'\
                    '4. A clear Markdown List of ALL files to be created, including their full paths.',
    agent=software_architect,
    context=[requirement_analysis_task] # Pass output from analyst task
)

### step 5: custom file write tool
A specialized tool to save the generated code and architecture files to local storage.

In [7]:
# 5. Define FileWriteTool
from crewai.tools import BaseTool

class FileWriteTool(BaseTool):
    name: str = "File Write Tool"
    description: str = "Useful to write content to a file. Input should be a dictionary with 'filename' and 'content' keys."

    def _run(self, filename: str, content: str) -> str:
        try:
            import os
            directory = os.path.dirname(filename)
            if directory:
                os.makedirs(directory, exist_ok=True)
            with open(filename, 'w') as f:
                f.write(content)
            return f"File {filename} written successfully."
        except Exception as e:
            return f"Error writing file: {e}"

file_write_tool = FileWriteTool()


### step 6: define backend developer agent & task
The developer agent writes the production-ready code based on the architecture specs.

In [8]:
# 6. Define Code Generation Agent
code_generation_agent = Agent(
    role='Senior Backend Developer',
    goal='Write production-ready modular code based on the architecture',
    backstory='You are a senior developer who writes clean, testable, and maintainable code. You follow best practices and design patterns.',
    verbose=True,
    allow_delegation=False,
    tools=[file_write_tool]
)


In [9]:
# 7. Define Code Generation Task
code_generation_task = Task(
    description='Generate the code and related files based on the architecture and Folder Structure provided by the Software Architect.\n'\
                '1. First, review the "File and Module Breakdown" and the "Markdown List of ALL files" from the Architect\'s output.\n'\
                '2. Create a root folder to the project then create all the files in the root folder.'
                '3. Iterate through EVERY file in that list. Do not skip any files.\n'\
                '4. For each file, generate the code and use the FileWriteTool to save it to the disk.\n'\
                '5. Strictly follow the folder structure and create the files and paths accordingly starting from the root directory.\n'\
                '6. Double-check against the list to ensure no files were skipped.\n'\
                'Ensure the code is production-ready, modular, and follows the recommended tech stack.',
    expected_output='Production-ready code saved to files. A list of files created.',
    agent=code_generation_agent,
    context=[architecture_task]
)


### step 7: define qa engineer agent & task
Ensures code quality by generating and running a comprehensive suite of unit and integration tests.

In [10]:
# 9. Define Test Generation Agent
test_generation_agent = Agent(
    role='QA Automation Engineer',
    goal='Generate comprehensive unit and integration tests with strict coverage',
    backstory='You are a quality-obsessed engineer who catches edge cases and ensures code robustness. You focus on high code coverage.',
    verbose=True,
    allow_delegation=False,
    tools=[file_write_tool]
)


In [11]:
# 10. Define Test Generation Task
test_generation_task = Task(
    description='Based on the code generated by the Senior Backend Developer, create a comprehensive test suite.\n'\
                '1. Analyze the code files generated in the previous task.\n'\
                '2. Create a separate "tests/" directory.\n'\
                '3. For EACH code file, generate a corresponding test file (e.g., test_user.py for user.py).\n'\
                '4. Write unit tests using unittest or pytest to cover happy paths and edge cases.\n'\
                '5. Create mock objects for any external dependencies.\n'\
                'Use the FileWriteTool to save the test files to the "tests/" directory.',
    expected_output='A set of test files (unit and integration tests) saved in the tests/ directory.',
    agent=test_generation_agent,
    context=[code_generation_task] # Pass output from code gen task
)


### step 8: custom word doc writer tool
Generates professional-grade technical documentation in Word format.

In [12]:
# 12. Define WordWriterTool
from crewai.tools import BaseTool

class WordWriterTool(BaseTool):
    name: str = "Word Writer Tool"
    description: str = "Useful to write content to a Word (.docx) file. Input should be a dictionary with 'filename' and 'content' keys. 'content' should be a string (markdown-like is fine, I will format headers)."

    def _run(self, filename: str, content: str) -> str:
        try:
            from docx import Document
            import os
            
            doc = Document()
            
            # Simple Markdown-ish parser to docx
            lines = content.split('\n')
            for line in lines:
                stripped = line.strip()
                if stripped.startswith('# '):
                    doc.add_heading(stripped[2:], level=1)
                elif stripped.startswith('## '):
                    doc.add_heading(stripped[3:], level=2)
                elif stripped.startswith('### '):
                    doc.add_heading(stripped[4:], level=3)
                elif stripped.startswith('- ') or stripped.startswith('* '):
                    doc.add_paragraph(stripped[2:], style='List Bullet')
                else:
                    doc.add_paragraph(line)
            
            directory = os.path.dirname(filename)
            if directory:
                os.makedirs(directory, exist_ok=True)
            
            doc.save(filename)
            return f"File {filename} written successfully."
        except Exception as e:
            return f"Error writing file: {e}"

word_write_tool = WordWriterTool()


### step 9: define technical documentation agent & task
Finalizes the delivery cycle by collating all work into a unified documentation file.

In [13]:
# 13. Define Documentation Agent
documentation_agent = Agent(
    role='Technical Documentation Specialist',
    goal='Produce comprehensive technical documentation in .docx format',
    backstory='You are a specialist in creating clear, structured technical documentation for developers and stakeholders.',
    verbose=True,
    allow_delegation=False,
    tools=[word_write_tool]
)


In [14]:
# 14. Define Documentation Task
documentation_task = Task(
    description='Generate a SINGLE comprehensive technical documentation document based on the codebase and architecture.\n'\
                'The document MUST be saved as "docs/technical_documentation.docx".\n'\
                'Include the following sections:\n'\
                '1. Project Overview\n'\
                '2. System Architecture (High-level)\n'\
                '3. Tech Stack\n'\
                '4. API Documentation (Endpoints, methods, params)\n'\
                '5. Setup & Installation Instructions\n'\
                '6. Usage Examples\n'\
                '7. Testing Guide\n'\
                'Use the WordWriterTool to save the file. Format the content with Markdown-style headers (#, ##, ###) and lists (- ) so the tool can format it correctly.',
    expected_output='A single .docx file named technical_documentation.docx containing all sections.',
    agent=documentation_agent,
    context=[requirement_analysis_task, architecture_task, code_generation_task, test_generation_task]
)


### step 10: assemble & kickoff the delivery crew
Executes the end-to-end software delivery pipeline sequentially.

In [15]:
# 11. Create the Crew
tech_crew = Crew(
    agents=[requirement_analyst, software_architect, code_generation_agent, test_generation_agent, documentation_agent],
    tasks=[requirement_analysis_task, architecture_task, code_generation_task, test_generation_task, documentation_task],
    process=Process.sequential,
    verbose=True
)


In [16]:
# 6. Run the Crew
print("Welcome to the AI Software Delivery System")
#feature = input("Enter the feature request: ")
feature = "Build a simple one page website for HCL with css and html."
if feature:
    result = tech_crew.kickoff(inputs={'feature_request': feature})
    print("\n\n########################")
    print("## Final Result ##")
    print("########################\n")
    print(result)
else:
    print("No feature request provided.")

Welcome to the AI Software Delivery System


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  130a4b23-a393-47f4-8b64-82906d031042                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following feature request and generate a detailed technical specification: Build a simple    │
│  one page website for HCL with css and html.                                                                    │
│  ID: 17e46533-28ff-46fb-89ed-1f23cc533a87                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Product Analyst                                                                                         │
│                                                                                                                 │
│  Task: Analyze the following feature request and generate a detailed technical specification: Build a simple    │
│  one page website for HCL with css and html.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Product Analyst                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Technical Specification Document                                                                               │
│                                                                                                                 │
│  Feature Request: Build a simple one-page website for HCL using HTML and CSS                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  1. Functional Requirements                                                                                     │
│                                                                                                                 │
│  1.1 Homepage Layout                                                                                            │
│  - The website shall be composed of a single page accessible via a root URL (e.g., www.hcl.com).                │
│  - The page shall contain the following sections arranged vertically in order:                                  │
│    a) Header with company logo and navigation menu (with anchor links scrolling smoothly to each section)       │
│    b) Hero section with a prominent headline, short descriptive tagline, and a call-to-action button            │
│    c) About Us section containing a brief description of HCL’s mission and services                             │
│    d) Services section listing key services offered by HCL in a clean, succinct format                          │
│    e) Contact section with contact details (address, phone, email) and a simple contact form (fields: name,     │
│  email, message + submit button)                                                                                │
│    f) Footer with copyright information and links to social media profiles                                      │
│                                                                                                                 │
│  1.2 Navigation Menu                                                                                            │
│  - The header navigation menu items shall link to the page sections using smooth scrolling on click.            │
│  - The navigation menu shall remain visible on scrolling (sticky header).                                       │
│                                                                                                                 │
│  1.3 Responsive Design                                                                                          │
│  - The website shall adapt layout gracefully across devices: desktop, tablet, mobile.                           │
│  - On smaller screens, the navigation menu should collapse to a hamburger menu.                                 │
│                                                                                                                 │
│  1.4 Contact Form                                                                                               │
│  - Basic client-side validation on required fields (name, email format).                                        │
│  - Form submission shall not require backend integratio

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Documentation Specialist                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  A single comprehensive technical documentation document named "technical_documentation.docx" has been created  │
│  in "docs/" directory. It includes all required sections with clear, structured content formatted in Markdown   │
│  style for headers and lists, ready for proper Word formatting. This meets the specified requirements.          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The file "docs/technical_documentation.docx" contains the complete technical documentation as requested.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Generate a SINGLE comprehensive technical documentation document based on the codebase and architecture.       │
│  The document MUST be saved as "docs/technical_documentation.docx".                                             │
│  Include the following sections:                                                                                │
│  1. Project Overview                                                                                            │
│  2. System Architecture (High-level)                                                                            │
│  3. Tech Stack                                                                                                  │
│  4. API Documentation (Endpoints, methods, params)                                                              │
│  5. Setup & Installation Instructions                                                                           │
│  6. Usage Examples                                                                                              │
│  7. Testing Guide                                                                                               │
│  Use the WordWriterTool to save the file. Format the content with Markdown-style headers (#, ##, ###) and       │
│  lists (- ) so the tool can format it correctly.                                                                │
│  Agent:                                                                                                         │
│  Technical Documentation Specialist                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  130a4b23-a393-47f4-8b64-82906d031042                                                                           │
│  Final Output: A single comprehensive technical documentation document named "technical_documentation.docx"     │
│  has been created in "docs/" directory. It includes all required sections with clear, structured content        │
│  formatted in Markdown style for headers and lists, ready for proper Word formatting. This meets the specified  │
│  requirements.                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The file "docs/technical_documentation.docx" contains the complete technical documentation as requested.       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



########################
## Final Result ##
########################

A single comprehensive technical documentation document named "technical_documentation.docx" has been created in "docs/" directory. It includes all required sections with clear, structured content formatted in Markdown style for headers and lists, ready for proper Word formatting. This meets the specified requirements.

Final Answer:
The file "docs/technical_documentation.docx" contains the complete technical documentation as requested.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯